//sdes


In [ ]:
#include <iostream>
#include <vector>
#include <string>
using namespace std;

// ---------- Utility Helpers ----------

vector<int> permute(const vector<int>& bits, const vector<int>& perm) {
    vector<int> out(perm.size());
    for (size_t i = 0; i < perm.size(); ++i)
        out[i] = bits[perm[i] - 1];   // 1-based indexing
    return out;
}

vector<int> leftShift5(const vector<int>& fiveBits, int shifts) {
    vector<int> out(5);
    for (int i = 0; i < 5; ++i)
        out[i] = fiveBits[(i + shifts) % 5];
    return out;
}

vector<int> concat(const vector<int>& a, const vector<int>& b) {
    vector<int> out = a;
    out.insert(out.end(), b.begin(), b.end());
    return out;
}

vector<int> xorVec(const vector<int>& a, const vector<int>& b) {
    vector<int> out(a.size());
    for (size_t i = 0; i < a.size(); ++i)
        out[i] = a[i] ^ b[i];
    return out;
}

void split4(const vector<int>& eight, vector<int>& L, vector<int>& R) {
    L.assign(eight.begin(), eight.begin() + 4);
    R.assign(eight.begin() + 4, eight.end());
}

vector<int> intToBits2(int x) {
    return { (x >> 1) & 1, x & 1 };
}

// ---------- S-DES CONSTANTS ----------

const vector<int> P10 = {3,5,2,7,4,10,1,9,8,6};
const vector<int> P8  = {6,3,7,4,8,5,10,9};
const vector<int> IP  = {2,6,3,1,4,8,5,7};
const vector<int> IPI = {4,1,3,5,7,2,8,6};
const vector<int> EP  = {4,1,2,3,2,3,4,1};
const vector<int> P4  = {2,4,3,1};

const int S0[4][4] = {
    {1,0,3,2},
    {3,2,1,0},
    {0,2,1,3},
    {3,1,3,2}
};

const int S1[4][4] = {
    {0,1,2,3},
    {2,0,1,3},
    {3,0,1,0},
    {2,1,0,3}
};

// ---------- Key Generation ----------

pair<vector<int>, vector<int>> generateSubkeys(const vector<int>& key10) {

    vector<int> p10 = permute(key10, P10);

    vector<int> left5(p10.begin(), p10.begin() + 5);
    vector<int> right5(p10.begin() + 5, p10.end());

    left5  = leftShift5(left5, 1);
    right5 = leftShift5(right5, 1);

    vector<int> K1 = permute(concat(left5, right5), P8);

    left5  = leftShift5(left5, 2);
    right5 = leftShift5(right5, 2);

    vector<int> K2 = permute(concat(left5, right5), P8);

    return {K1, K2};
}

// ---------- F Function ----------

vector<int> sboxLookup(const int S[4][4], const vector<int>& in4) {
    int row = (in4[0] << 1) | in4[3];
    int col = (in4[1] << 1) | in4[2];
    return intToBits2(S[row][col]);
}

vector<int> F(const vector<int>& right4, const vector<int>& subkey8) {

    vector<int> er = permute(right4, EP);
    vector<int> x  = xorVec(er, subkey8);

    vector<int> xL(x.begin(), x.begin() + 4);
    vector<int> xR(x.begin() + 4, x.end());

    vector<int> s0out = sboxLookup(S0, xL);
    vector<int> s1out = sboxLookup(S1, xR);

    return permute(concat(s0out, s1out), P4);
}

vector<int> fk(const vector<int>& eight, const vector<int>& subkey8) {
    vector<int> L, R;
    split4(eight, L, R);

    vector<int> newL = xorVec(L, F(R, subkey8));
    return concat(newL, R);
}

vector<int> SW(const vector<int>& eight) {
    vector<int> L, R;
    split4(eight, L, R);
    return concat(R, L);
}

// ---------- Encrypt / Decrypt ----------

vector<int> encrypt(const vector<int>& plaintext8,
                    const vector<int>& key10) {

    auto subkeys = generateSubkeys(key10);
    vector<int> state = permute(plaintext8, IP);

    state = fk(state, subkeys.first);
    state = SW(state);
    state = fk(state, subkeys.second);

    return permute(state, IPI);
}

vector<int> decrypt(const vector<int>& ciphertext8,
                    const vector<int>& key10) {

    auto subkeys = generateSubkeys(key10);
    vector<int> state = permute(ciphertext8, IP);

    state = fk(state, subkeys.second);
    state = SW(state);
    state = fk(state, subkeys.first);

    return permute(state, IPI);
}

// ---------- I/O Helpers ----------

vector<int> readBitsString(const string& s, int expectedLen) {
    vector<int> bits;

    for (char c : s)
        if (c == '0' || c == '1')
            bits.push_back(c - '0');

    if ((int)bits.size() != expectedLen) {
        cerr << "Error: expected "
             << expectedLen << " bits.\n";
        exit(1);
    }
    return bits;
}

void printBits(const vector<int>& bits) {
    for (int b : bits) cout << b;
    cout << endl;
}

// ---------- MAIN ----------

int main() {
    string ptStr, keyStr;

    cout << "Enter 8-bit plaintext: ";
    cin >> ptStr;

    cout << "Enter 10-bit key: ";
    cin >> keyStr;

    vector<int> plaintext = readBitsString(ptStr, 8);
    vector<int> key       = readBitsString(keyStr, 10);

    vector<int> ciphertext = encrypt(plaintext, key);
    cout << "Ciphertext: ";
    printBits(ciphertext);

    vector<int> decrypted = decrypt(ciphertext, key);
    cout << "Decrypted: ";
    printBits(decrypted);

    return 0;
}

//rsa

#include <iostream>
#include <cmath>
using namespace std;

// Fast modular exponentiation
long long power(long long base, long long exp, long long mod) {
    long long res = 1;
    base = base % mod;

    while (exp > 0) {
        if (exp % 2 == 1)
            res = (res * base) % mod;

        base = (base * base) % mod;
        exp = exp / 2;
    }
    return res;
}

// Function to find modular inverse using Extended Euclidean Algorithm
long long modInverse(long long e, long long phi) {
    long long t = 0, newt = 1;
    long long r = phi, newr = e;

    while (newr != 0) {
        long long quotient = r / newr;

        t = t - quotient * newt;
        swap(t, newt);

        r = r - quotient * newr;
        swap(r, newr);
    }

    if (r > 1) return -1;   // Inverse does not exist
    if (t < 0) t = t + phi;

    return t;
}

int main() {
    long long p, q, e, M;

    cin >> p >> q >> e >> M;

    // 1. Calculate n
    long long n = p * q;

    // 2. Calculate Totient phi(n)
    long long phi = (p - 1) * (q - 1);

    // 3. Find private key d
    long long d = modInverse(e, phi);

    // 4. Encrypt message
    long long C = power(M, e, n);

    // Output Results
    cout << "--- RSA Results ---" << endl;
    cout << "p: " << p << ", q: " << q << endl;
    cout << "n (Modulus): " << n << endl;
    cout << "phi(n) (Totient): " << phi << endl;
    cout << "e (Public Key): " << e << endl;
    cout << "d (Private Key): " << d << endl;
    cout << "-------------------" << endl;

    cout << "Original Message M: " << M << endl;
    cout << "Encrypted Cipher C: " << C << endl;

    // 5. Decryption check
    long long M_ = power(C, d, n);

    if (M == M_)
        cout << "Encryption and Decryption was successful";
    else
        cout << "Encryption and Decryption was not successful";

    return 0;
}

//AES

#include <iostream>
#include <iomanip>
#include <vector>
#include <cstring>

using namespace std;

#define Nb 4   

int Nk;  
int Nr; 
//                  SBOX 
unsigned char sbox[256] = {
0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
};

unsigned char inv_sbox[256];
unsigned char Rcon[10] = {0x01,0x02,0x04,0x08,0x10,0x20,0x40,0x80,0x1B,0x36};

//                 GF Multiply 
unsigned char GFM(unsigned char a, unsigned char b){
    unsigned char p = 0;
    for(int i=0;i<8;i++){
        if(b&1) p^=a;
        bool hi = a & 0x80;
        a<<=1;
        if(hi) a^=0x1b;
        b>>=1;
    }
    return p;
}

//                 Key Expansion 
void KeyEx(vector<unsigned char>& key, int keyBytes){
    int i = keyBytes;
    int totalBytes = 16*(Nr+1);
    unsigned char temp[4];

    while(i<totalBytes){
        for(int j=0;j<4;j++)
            temp[j]=key[i-4+j];

        if(i%keyBytes==0){
            unsigned char t=temp[0];
            temp[0]=temp[1]; temp[1]=temp[2];
            temp[2]=temp[3]; temp[3]=t;

            for(int j=0;j<4;j++)
                temp[j]=sbox[temp[j]];

            temp[0]^=Rcon[(i/keyBytes)-1];
        }
        else if(Nk>6 && i%keyBytes==16){
            for(int j=0;j<4;j++)
                temp[j]=sbox[temp[j]];
        }

        for(int j=0;j<4;j++){
            key[i]=key[i-keyBytes]^temp[j];
            i++;
        }
    }
}

//                    AES Core 
void AddRK(unsigned char state[4][4], unsigned char* rk){
    for(int c=0;c<4;c++)
        for(int r=0;r<4;r++)
            state[r][c]^=rk[c*4+r];
}

void SubB(unsigned char state[4][4]){
    for(int r=0;r<4;r++)
        for(int c=0;c<4;c++)
            state[r][c]=sbox[state[r][c]];
}

void ISubB(unsigned char state[4][4]){
    for(int r=0;r<4;r++)
        for(int c=0;c<4;c++)
            state[r][c]=inv_sbox[state[r][c]];
}

void ShR(unsigned char s[4][4]){
    unsigned char t;

    t=s[1][0]; s[1][0]=s[1][1]; s[1][1]=s[1][2]; s[1][2]=s[1][3]; s[1][3]=t;
    swap(s[2][0],s[2][2]); swap(s[2][1],s[2][3]);
    t=s[3][3]; s[3][3]=s[3][2]; s[3][2]=s[3][1]; s[3][1]=s[3][0]; s[3][0]=t;
}

void IShR(unsigned char s[4][4]){
    unsigned char t;

    t=s[1][3]; s[1][3]=s[1][2]; s[1][2]=s[1][1]; s[1][1]=s[1][0]; s[1][0]=t;
    swap(s[2][0],s[2][2]); swap(s[2][1],s[2][3]);
    t=s[3][0]; s[3][0]=s[3][1]; s[3][1]=s[3][2]; s[3][2]=s[3][3]; s[3][3]=t;
}

void MixC(unsigned char s[4][4]){
    unsigned char t[4];
    for(int c=0;c<4;c++){
        t[0]=GFM(2,s[0][c])^GFM(3,s[1][c])^s[2][c]^s[3][c];
        t[1]=s[0][c]^GFM(2,s[1][c])^GFM(3,s[2][c])^s[3][c];
        t[2]=s[0][c]^s[1][c]^GFM(2,s[2][c])^GFM(3,s[3][c]);
        t[3]=GFM(3,s[0][c])^s[1][c]^s[2][c]^GFM(2,s[3][c]);
        for(int r=0;r<4;r++) s[r][c]=t[r];
    }
}

void IMixC(unsigned char s[4][4]){
    unsigned char t[4];
    for(int c=0;c<4;c++){
        t[0]=GFM(14,s[0][c])^GFM(11,s[1][c])^GFM(13,s[2][c])^GFM(9,s[3][c]);
        t[1]=GFM(9,s[0][c])^GFM(14,s[1][c])^GFM(11,s[2][c])^GFM(13,s[3][c]);
        t[2]=GFM(13,s[0][c])^GFM(9,s[1][c])^GFM(14,s[2][c])^GFM(11,s[3][c]);
        t[3]=GFM(11,s[0][c])^GFM(13,s[1][c])^GFM(9,s[2][c])^GFM(14,s[3][c]);
        for(int r=0;r<4;r++) s[r][c]=t[r];
    }
}

//                    MAIN 
int main(){

    for(int i=0;i<256;i++)
        inv_sbox[sbox[i]] = i;

    string plaintext;
    cout<<"Enter Plaintext: ";
    getline(cin,plaintext);

    string keystr;
    cout<<"Enter Key (16/24/32 chars): ";
    getline(cin,keystr);

    int keyBytes = keystr.size();

    if(keyBytes==16){ Nk=4; Nr=10; }
    else if(keyBytes==24){ Nk=6; Nr=12; }
    else if(keyBytes==32){ Nk=8; Nr=14; }
    else{
        cout<<"Key must be 16, 24 or 32 characters!\n";
        return 0;
    }

    vector<unsigned char> key(16*(Nr+1));
    memcpy(key.data(),keystr.c_str(),keyBytes);
    KeyEx(key,keyBytes);

    //                   PKCS7 Padding
    int pad=16-(plaintext.size()%16);
    for(int i=0;i<pad;i++)
        plaintext.push_back((char)pad);

    vector<unsigned char> data(plaintext.begin(),plaintext.end());
    vector<unsigned char> iv(16,0x00);

    //                   ENCRYPTION  
    for(size_t i=0;i<data.size();i+=16){

        for(int j=0;j<16;j++)
            data[i+j]^=iv[j];

        unsigned char state[4][4];
        for(int r=0;r<4;r++)
            for(int c=0;c<4;c++)
                state[r][c]=data[i+c*4+r];

        AddRK(state,key.data());

        for(int round=1;round<Nr;round++){
            SubB(state);
            ShR(state);
            MixC(state);
            AddRK(state,key.data()+round*16);
        }

        SubB(state);
        ShR(state);
        AddRK(state,key.data()+Nr*16);

        for(int r=0;r<4;r++)
            for(int c=0;c<4;c++)
                data[i+c*4+r]=state[r][c];

        iv.assign(data.begin()+i,data.begin()+i+16);
    }

    cout<<"\nCiphertext (Hex):\n";
    for(auto b:data)
        cout<<hex<<setw(2)<<setfill('0')<<(int)b;
    cout<<dec<<"\n";

    //                    DECRYPTION 
    iv.assign(16,0x00);
    vector<unsigned char> decrypted=data;

    for(size_t i=0;i<decrypted.size();i+=16){

        unsigned char state[4][4];
        for(int r=0;r<4;r++)
            for(int c=0;c<4;c++)
                state[r][c]=decrypted[i+c*4+r];

        AddRK(state,key.data()+Nr*16);

        for(int round=Nr-1;round>0;round--){
            IShR(state);
            ISubB(state);
            AddRK(state,key.data()+round*16);
            IMixC(state);
        }

        IShR(state);
        ISubB(state);
        AddRK(state,key.data());

        for(int r=0;r<4;r++)
            for(int c=0;c<4;c++)
                decrypted[i+c*4+r]=state[r][c]^iv[c*4+r];

        iv.assign(data.begin()+i,data.begin()+i+16);
    }

    int padding=decrypted.back();
    decrypted.resize(decrypted.size()-padding);

    cout<<"\nDecrypted Plaintext:\n";
    for(auto c:decrypted)
        cout<<(char)c;

    cout<<"\n\nStatus: Decryption Successful\n";

    return 0;
}


//diffie-hellman


#include <bits/stdc++.h>
using namespace std;

// Function to perform modular multiplication safely
long long mulmod(long long a, long long b, long long mod) {
    long long res = 0;
    a %= mod;
    while (b > 0) {
        if (b & 1) res = (res + a) % mod;
        a = (a * 2) % mod;
        b >>= 1;
    }
    return res;
}

// Function to perform modular exponentiation
long long power(long long base, long long exp, long long mod) {
    long long res = 1;
    base %= mod;
    while (exp > 0) {
        if (exp & 1) res = mulmod(res, base, mod);
        base = mulmod(base, base, mod);
        exp >>= 1;
    }
    return res;
}

int main() {
    long long q, alpha;
    long long a, b; // private keys

    cout << "Enter q (prime number): ";
    cin >> q;
    cout << "Enter alpha (primitive root of q): ";
    cin >> alpha;
    cout << "Enter Alice's private key (a): ";
    cin >> a;
    cout << "Enter Bob's private key (b): ";
    cin >> b;

    // Public keys
    long long A = power(alpha, a, q);
    long long B = power(alpha, b, q);

    // Shared secret keys
    long long keyA = power(B, a, q);
    long long keyB = power(A, b, q);

    cout << "\nStep 1: Public parameters" << endl;
    cout << "q = " << q << ", alpha = " << alpha << endl;

    cout << "\nStep 2: Private keys" << endl;
    cout << "Alice private key (a) = " << a << endl;
    cout << "Bob private key (b) = " << b << endl;

    cout << "\nStep 3: Public keys" << endl;
    cout << "Alice public key A = alpha^a mod q = " << A << endl;
    cout << "Bob public key B = alpha^b mod q = " << B << endl;

    cout << "\nStep 4: Shared secret computation" << endl;
    cout << "Alice computes K = B^a mod q = " << keyA << endl;
    cout << "Bob computes K = A^b mod q = " << keyB << endl;

    cout << "\nFinal Conclusion:" << endl;
    if (keyA == keyB)
        cout << "Shared secret keys match. Diffie-Hellman key exchange successful." << endl;
    else
        cout << "Shared secret keys do not match. Key exchange failed." << endl;

    return 0;
}

//man in the middle

#include <iostream>
using namespace std;

// Fast modular exponentiation
long long power(long long base, long long exp, long long mod) {
    long long result = 1;
    base %= mod;

    while (exp > 0) {
        if (exp & 1)
            result = (result * base) % mod;

        exp >>= 1;
        base = (base * base) % mod;
    }
    return result;
}

int main() {
    long long q, alpha;
    long long XA, XB;
    long long XD1, XD2; // Darth's two private keys

    cout << "Enter prime number q: ";
    cin >> q;

    cout << "Enter primitive root alpha: ";
    cin >> alpha;

    cout << "Enter Alice private key XA: ";
    cin >> XA;

    cout << "Enter Bob private key XB: ";
    cin >> XB;

    cout << "Enter Darth private key towards Bob (XD1): ";
    cin >> XD1;

    cout << "Enter Darth private key towards Alice (XD2): ";
    cin >> XD2;

    cout << "\n--- Public Key Generation ---\n";

    long long YA = power(alpha, XA, q);
    long long YB = power(alpha, XB, q);
    long long YD1 = power(alpha, XD1, q);
    long long YD2 = power(alpha, XD2, q);

    cout << "Alice Public Key YA = " << YA << endl;
    cout << "Bob Public Key YB = " << YB << endl;
    cout << "Darth Public Key YD1 = " << YD1 << " (sent to Bob)\n";
    cout << "Darth Public Key YD2 = " << YD2 << " (sent to Alice)\n";

    cout << "\n--- Interception Phase ---\n";
    cout << "Alice sends YA = " << YA
         << " -> Darth intercepts -> sends YD1 = " << YD1 << " to Bob\n";

    cout << "Bob sends YB = " << YB
         << " -> Darth intercepts -> sends YD2 = " << YD2 << " to Alice\n";

    cout << "\n--- Shared Key Computation ---\n";

    long long K_Alice = power(YD2, XA, q);
    long long K_Bob   = power(YD1, XB, q);
    long long K_Darth_A = power(YA, XD2, q);
    long long K_Darth_B = power(YB, XD1, q);

    cout << "Alice computes key: K2 = (YD2^XA) mod q = "
         << K_Alice << endl;

    cout << "Bob computes key: K1 = (YD1^XB) mod q = "
         << K_Bob << endl;

    cout << "\nDarth computes with Alice: K2 = (YA^XD2) mod q = "
         << K_Darth_A << endl;

    cout << "Darth computes with Bob: K1 = (YB^XD1) mod q = "
         << K_Darth_B << endl;

    cout << "\n--- Final Shared Keys ---\n";
    cout << "Alice <-> Darth shared key: " << K_Alice << endl;
    cout << "Bob <-> Darth shared key: " << K_Bob << endl;

    return 0;
}